In [98]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

In [99]:
df = pd.read_csv("../datasets/tutorials.csv")


In [100]:
type(df)

pandas.core.frame.DataFrame

In [101]:
df.head()

,Video_Title,Video_Description,Upload_Date,Duration_Minutes,View_Count,Like_Count,Comment_Count,Channel_Subscriber_Count,Actual_Quality_Score
0,Learn Python in 10 Minutes!,Super fast Python tutorial for absolute beginn...,2023-01-15,10,980000,12000,800,150000,28
1,Complete Python Bootcamp 2024,Comprehensive Python course covering all funda...,2023-02-10,480,520000,41000,3200,210000,88
2,Machine Learning Crash Course,Everything about ML in 5 minutes. Covers neura...,2023-01-20,5,1200000,9500,600,300000,18
3,Deep Learning with PyTorch,Step by step PyTorch tutorial with real projec...,2023-03-05,360,280000,24000,1800,180000,85
4,Web Dev in 1 Hour,Full stack web development explained in just 6...,2023-01-25,60,870000,11000,750,220000,32


In [102]:
df.shape

(138, 9)

In [103]:
df['like_to_view_ratio'] = df['Like_Count'] / df['View_Count']

In [104]:
df['comment_to_view_ratio'] = df['Comment_Count'] / df['View_Count']

In [105]:
df['engagement_rate'] = (df['Like_Count'] + df['Comment_Count']) / df['View_Count']

In [106]:
df['title_length'] = df['Video_Title'].apply(len)

In [107]:
df.tail()

,Video_Title,Video_Description,Upload_Date,Duration_Minutes,View_Count,Like_Count,Comment_Count,Channel_Subscriber_Count,Actual_Quality_Score,like_to_view_ratio,comment_to_view_ratio,engagement_rate,title_length
133,Distributed Systems,CAP theorem consistency availability partition...,2023-06-30,480,58000,6200,490,43000,92,0.106897,0.008448,0.115345,19
134,Unit Testing in Python,Write tests with pytest. Fixtures mocking para...,2023-05-10,220,118000,11000,850,80000,81,0.093220,0.007203,0.100424,22
135,Message Queues and RabbitMQ,Asynchronous messaging with RabbitMQ. Exchange...,2023-07-16,260,78000,7600,590,57000,80,0.097436,0.007564,0.105000,27
136,Coding at 3AM and It Works,Vlog of a programmer debugging at 3am. Relatab...,2023-02-24,15,1400000,22000,1700,360000,16,0.015714,0.001214,0.016929,26
137,Information Retrieval Systems,Search engines from scratch. Inverted index TF...,2023-07-26,420,45000,5000,390,33000,90,0.111111,0.008667,0.119778,29


In [108]:
clickbait_words = ['minute', 'overnight', 'instantly', 'secret', 'trick', 'rich', 'cheat', 'replace', 'blow']

In [109]:
df['is_clickbait'] = df['Video_Title'].str.lower().apply(
    lambda title: int(any(word in title for word in clickbait_words))
)

In [110]:
df["is_clickbait"].value_counts()

is_clickbait
0    124
1     14
Name: count, dtype: int64

features = the columns we give to the model to learn from

target = the column we want the model to predict (Actual_Quality_Score)

In [111]:
features = [
    'Duration_Minutes',
    'View_Count',
    'Like_Count',
    'Comment_Count',
    'Channel_Subscriber_Count',
    'like_to_view_ratio',
    'comment_to_view_ratio',
    'engagement_rate',
    'title_length',
    'is_clickbait'
]

xfeatures = df[features]
y = df['Actual_Quality_Score']



In [112]:
xfeatures.head()

,Duration_Minutes,View_Count,Like_Count,Comment_Count,Channel_Subscriber_Count,like_to_view_ratio,comment_to_view_ratio,engagement_rate,title_length,is_clickbait
0,10,980000,12000,800,150000,0.012245,0.000816,0.013061,27,1
1,480,520000,41000,3200,210000,0.078846,0.006154,0.085000,29,0
2,5,1200000,9500,600,300000,0.007917,0.000500,0.008417,29,0
3,360,280000,24000,1800,180000,0.085714,0.006429,0.092143,26,0
4,60,870000,11000,750,220000,0.012644,0.000862,0.013506,17,0



I split the data into:
- Training set (80%) — the model learns from this
- Test set (20%) — we check how well the model does on unseen data

In [113]:
X_train, X_test, y_train, y_test = train_test_split(xfeatures, y, test_size=0.2,random_state=42)

In [114]:
X_train

,Duration_Minutes,View_Count,Like_Count,Comment_Count,Channel_Subscriber_Count,like_to_view_ratio,comment_to_view_ratio,engagement_rate,title_length,is_clickbait
10,240,180000,16000,1200,120000,0.088889,0.006667,0.095556,24,0
125,380,125000,11500,880,83000,0.092000,0.007040,0.099040,31,0
51,360,270000,24000,1820,156000,0.088889,0.006741,0.095630,22,0
11,540,220000,19000,1500,130000,0.086364,0.006818,0.093182,30,0
127,360,62000,6500,510,46000,0.104839,0.008226,0.113065,27,0
...,...,...,...,...,...,...,...,...,...,...
71,310,130000,11500,890,87000,0.088462,0.006846,0.095308,29,0
106,25,1300000,20000,1500,340000,0.015385,0.001154,0.016538,31,0
14,15,950000,6000,350,240000,0.006316,0.000368,0.006684,24,1
92,260,110000,10200,790,76000,0.092727,0.007182,0.099909,26,0


In [115]:
X_test

,Duration_Minutes,View_Count,Like_Count,Comment_Count,Channel_Subscriber_Count,like_to_view_ratio,comment_to_view_ratio,engagement_rate,title_length,is_clickbait
89,340,95000,9000,700,70000,0.094737,0.007368,0.102105,18,0
105,340,85000,8200,640,63000,0.096471,0.007529,0.104000,27,0
12,30,2000000,14000,900,500000,0.007000,0.000450,0.007450,24,0
26,350,160000,14500,1100,108000,0.090625,0.006875,0.097500,23,0
64,180,225000,20000,1510,145000,0.088889,0.006711,0.095600,20,0
67,380,92000,8800,690,68000,0.095652,0.007500,0.103152,32,0
65,20,1900000,14000,900,480000,0.007368,0.000474,0.007842,24,1
117,300,72000,7200,560,54000,0.100000,0.007778,0.107778,22,0
108,240,165000,14800,1120,110000,0.089697,0.006788,0.096485,24,0
73,60,450000,22000,1700,195000,0.048889,0.003778,0.052667,29,0


In [116]:
y_test

89     82
105    85
12      8
26     78
64     85
67     87
65     13
117    76
108    78
73     58
85     74
45     82
76     11
68     48
31      9
42     60
27     82
19     91
86     58
56     14
109    83
4      32
111    35
36     77
104    65
136    16
16     22
18     55
Name: Actual_Quality_Score, dtype: int64

In [117]:
X_train.shape, X_test.shape

((110, 10), (28, 10))

Train Model 1: Linear Regression

In [118]:
model = LinearRegression()

In [119]:
model.fit(X_train, y_train)

LinearRegression()

In [120]:
ypred = model.predict(X_test)
ypred

array([81.11985833, 82.56278477,  8.90113048, 81.74557618, 80.42519677,
       83.46626676, 10.12118424, 82.85374168, 79.94166435, 58.3059035 ,
       78.24455727, 81.28564165, 12.57069946, 39.26899768,  9.69551401,
       53.40426571, 81.06978825, 87.74179958, 62.3755969 , 17.08307752,
       83.23425853, 27.31897469, 26.23630952, 78.77835729, 74.85726664,
       27.48375878, 19.07350933, 43.29027223])

In [121]:
y_test

89     82
105    85
12      8
26     78
64     85
67     87
65     13
117    76
108    78
73     58
85     74
45     82
76     11
68     48
31      9
42     60
27     82
19     91
86     58
56     14
109    83
4      32
111    35
36     77
104    65
136    16
16     22
18     55
Name: Actual_Quality_Score, dtype: int64

In [122]:
r2  = round(r2_score(y_test, ypred), 3)


In [123]:
r2

0.966

In [124]:
mae = round(mean_absolute_error(y_test,ypred), 2)

In [125]:
mae

np.float64(4.06)

Train Model 2: Random Forest

In [126]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [127]:
rf_preds = rf_model.predict(X_test)


In [128]:
X_test

,Duration_Minutes,View_Count,Like_Count,Comment_Count,Channel_Subscriber_Count,like_to_view_ratio,comment_to_view_ratio,engagement_rate,title_length,is_clickbait
89,340,95000,9000,700,70000,0.094737,0.007368,0.102105,18,0
105,340,85000,8200,640,63000,0.096471,0.007529,0.104000,27,0
12,30,2000000,14000,900,500000,0.007000,0.000450,0.007450,24,0
26,350,160000,14500,1100,108000,0.090625,0.006875,0.097500,23,0
64,180,225000,20000,1510,145000,0.088889,0.006711,0.095600,20,0
67,380,92000,8800,690,68000,0.095652,0.007500,0.103152,32,0
65,20,1900000,14000,900,480000,0.007368,0.000474,0.007842,24,1
117,300,72000,7200,560,54000,0.100000,0.007778,0.107778,22,0
108,240,165000,14800,1120,110000,0.089697,0.006788,0.096485,24,0
73,60,450000,22000,1700,195000,0.048889,0.003778,0.052667,29,0


In [129]:
rf_preds = rf_model.predict(X_test)


In [130]:
rf_r2  = round(r2_score(y_test, rf_preds), 3)
rf_mae = round(mean_absolute_error(y_test, rf_preds), 2)

In [131]:
comparison = pd.DataFrame({
    'Model' : ['Linear Regression', 'Random Forest'],
    'R2'    : [r2, rf_r2],
    'MAE'   : [mae, rf_mae]
})

comparison

,Model,R2,MAE
0,Linear Regression,0.966,4.06
1,Random Forest,0.974,3.50


R² tells us what fraction of the variation in quality the model explains.

R² = 1.0 would be perfect. R² close to 0 means the model is no better than just guessing the average.

MAE (Mean Absolute Error) tells us on average how many points off our predictions are.

##### Feature Importance

Random Forest tells us which features it relied on the most.

This answers: **what makes a tutorial clickbait vs genuinely good?**

In [132]:
importance = pd.Series(rf_model.feature_importances_, index=features)

In [133]:
importance = importance.sort_values(ascending=False).round(4)

In [134]:
importance

comment_to_view_ratio       0.2538
engagement_rate             0.2303
like_to_view_ratio          0.2285
View_Count                  0.1893
Duration_Minutes            0.0498
Channel_Subscriber_Count    0.0408
title_length                0.0028
Like_Count                  0.0023
Comment_Count               0.0023
is_clickbait                0.0001
dtype: float64

The most important feature for detecting a bad tutorial.

In [135]:
importance.idxmax()

'comment_to_view_ratio'

In [159]:
data = xfeatures.iloc[10]
data = np.array(data)
data = data.reshape(1,-1)

In [162]:
data.shape

(1, 10)

In [161]:
rf_model.predict(data)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


array([80.74])

In [136]:
df['predicted_score'] = rf_model.predict(xfeatures)


In [148]:
df.iloc[10]

Video_Title                                          Django REST API Tutorial
Video_Description           Build a complete REST API with Django. Authent...
Upload_Date                                                        2023-06-15
Duration_Minutes                                                          240
View_Count                                                             180000
Like_Count                                                              16000
Comment_Count                                                            1200
Channel_Subscriber_Count                                               120000
Actual_Quality_Score                                                       80
like_to_view_ratio                                                   0.088889
comment_to_view_ratio                                                0.006667
engagement_rate                                                      0.095556
title_length                                                    

In [137]:
df[['Video_Title', 'Actual_Quality_Score', 'predicted_score']]

,Video_Title,Actual_Quality_Score,predicted_score
0,Learn Python in 10 Minutes!,28,26.79
1,Complete Python Bootcamp 2024,88,80.80
2,Machine Learning Crash Course,18,17.73
3,Deep Learning with PyTorch,85,85.41
4,Web Dev in 1 Hour,32,30.01
...,...,...,...
133,Distributed Systems,92,91.31
134,Unit Testing in Python,81,81.33
135,Message Queues and RabbitMQ,80,80.67
136,Coding at 3AM and It Works,16,26.78


In [138]:
top10 = df.nlargest(10, 'predicted_score')[['Video_Title', 'predicted_score', 'Actual_Quality_Score', 'Duration_Minutes', 'engagement_rate']]

In [139]:
top10

,Video_Title,predicted_score,Actual_Quality_Score,Duration_Minutes,engagement_rate
114,Intro to Compilers,91.47,92,480,0.116875
133,Distributed Systems,91.31,92,480,0.115345
124,Probabilistic Graphical Models,90.81,91,440,0.123333
98,Statistical Learning Theory,90.70,91,420,0.113636
137,Information Retrieval Systems,90.45,90,420,0.119778
119,Mathematical Optimization,88.69,89,380,0.111552
25,Computer Networks Complete Course,88.38,90,420,0.103182
19,Operating Systems Fundamentals,88.30,91,480,0.108421
11,Algorithms and Data Structures,88.06,89,540,0.093182
74,Linear Algebra for Data Science,87.67,90,420,0.104091


In [140]:
top10 = top10.reset_index(drop=True)


In [141]:
top10.index = top10.index + 1
top10

,Video_Title,predicted_score,Actual_Quality_Score,Duration_Minutes,engagement_rate
1,Intro to Compilers,91.47,92,480,0.116875
2,Distributed Systems,91.31,92,480,0.115345
3,Probabilistic Graphical Models,90.81,91,440,0.123333
4,Statistical Learning Theory,90.70,91,420,0.113636
5,Information Retrieval Systems,90.45,90,420,0.119778
6,Mathematical Optimization,88.69,89,380,0.111552
7,Computer Networks Complete Course,88.38,90,420,0.103182
8,Operating Systems Fundamentals,88.30,91,480,0.108421
9,Algorithms and Data Structures,88.06,89,540,0.093182
10,Linear Algebra for Data Science,87.67,90,420,0.104091


In [142]:
bottom10 = df.nsmallest(10, 'predicted_score')[['Video_Title', 'predicted_score', 'Actual_Quality_Score', 'View_Count', 'like_to_view_ratio']]
bottom10 = bottom10.reset_index(drop=True)
bottom10.index = bottom10.index + 1
bottom10

,Video_Title,predicted_score,Actual_Quality_Score,View_Count,like_to_view_ratio
1,Interview Cheat Sheet One Page,6.99,5,2200000,0.007273
2,ChatGPT Will Replace Programmers,8.48,7,2300000,0.007391
3,Solve Leetcode While You Sleep,8.69,6,1550000,0.006452
4,Coding Will Make You Rich!,9.22,8,2100000,0.007143
5,Earn Money with AI Today,9.62,8,2000000,0.007000
6,Become 10x Programmer Instantly,9.96,10,2000000,0.007250
7,I Made $100K Freelancing,10.06,9,1800000,0.007222
8,Make $10000 with Coding,10.30,9,1900000,0.006842
9,Crack Any Coding Interview Tomorrow,10.60,10,1800000,0.006667
10,This One Line of Code Will Blow Your Mind,11.70,11,2100000,0.007619


- Worst tutorials have very **high view counts** but very **low like-to-view ratios**
- This is exactly the clickbait pattern — people click, watch for 30 seconds, then leave without liking

In [143]:
output = df[['Video_Title', 'Upload_Date', 'Duration_Minutes', 'View_Count',
             'Like_Count', 'Comment_Count', 'Channel_Subscriber_Count',
             'engagement_rate', 'like_to_view_ratio', 'is_clickbait',
             'Actual_Quality_Score', 'predicted_score']].copy()

In [144]:
output['Rank'] = output['predicted_score'].rank(ascending=False).astype(int)


In [145]:
output = output.sort_values('Rank')
output = output.reset_index(drop=True)

In [146]:
output.to_csv("tutorial_scores.csv", index=False)

In [147]:
output.head(15)

,Video_Title,Upload_Date,Duration_Minutes,View_Count,Like_Count,Comment_Count,Channel_Subscriber_Count,engagement_rate,like_to_view_ratio,is_clickbait,Actual_Quality_Score,predicted_score,Rank
0,Intro to Compilers,2023-06-26,480,48000,5200,410,36000,0.116875,0.108333,0,92,91.47,1
1,Distributed Systems,2023-06-30,480,58000,6200,490,43000,0.115345,0.106897,0,92,91.31,2
2,Probabilistic Graphical Models,2023-07-22,440,42000,4800,380,31000,0.123333,0.114286,0,91,90.81,3
3,Statistical Learning Theory,2023-07-12,420,55000,5800,450,41000,0.113636,0.105455,0,91,90.70,4
4,Information Retrieval Systems,2023-07-26,420,45000,5000,390,33000,0.119778,0.111111,0,90,90.45,5
5,Mathematical Optimization,2023-07-28,380,58000,6000,470,43000,0.111552,0.103448,0,89,88.69,6
6,Computer Networks Complete Course,2023-06-20,420,110000,10500,850,82000,0.103182,0.095455,0,90,88.38,7
7,Operating Systems Fundamentals,2023-06-01,480,95000,9500,800,75000,0.108421,0.100000,0,91,88.30,8
8,Algorithms and Data Structures,2023-03-20,540,220000,19000,1500,130000,0.093182,0.086364,0,89,88.06,9
9,Linear Algebra for Data Science,2023-06-25,420,88000,8500,660,65000,0.104091,0.096591,0,90,87.67,10
